## Create seasonal, multiannual SBAS networks from groups of Sentinel-1 burst data

This notebook focuses on using the `SBASNetwork` class to create SBAS networks of interferometric pairs from groups of adjacent Sentinel-1 burst data. Refer to the [SBASNetwork Tutorial](SBASNetwork.ipynb) for examples of creating SBASNetworks of networks of full Sentinel-1 scene or individual Sentinel-1 burst locations.

In [1]:
# This is useful if you are working with a dev install of asf_search and experimenting with changes to the codebase
%load_ext autoreload
%autoreload 2

### Create an asf_search.S1MultiBurstGroup object defining a group of Sentinel-1 bursts and subswaths

Refer to [HyP3's mutli-burst guidelines](https://hyp3-docs.asf.alaska.edu/guides/burst_insar_product_guide/#considerations-for-selecting-input-bursts) when assembling a collection of bursts.

In [2]:
import asf_search as asf

multiburst_group = asf.S1MultiBurstGroup(
    bursts=[
    asf.S1MultiBurst("173_370305", ("IW1", "IW2", "IW3")),
    asf.S1MultiBurst("173_370306", ("IW1", "IW2", "IW3")),
    asf.S1MultiBurst("173_370307", ("IW1", "IW2", "IW3"))
    ]
)
multiburst_group

S1MultiBurstGroup(bursts=[S1MultiBurst(relative_burst_id='173_370305', swaths=('IW1', 'IW2', 'IW3')), S1MultiBurst(relative_burst_id='173_370306', swaths=('IW1', 'IW2', 'IW3')), S1MultiBurst(relative_burst_id='173_370307', swaths=('IW1', 'IW2', 'IW3'))])

### Create a geoographic reference S1MultiBurstProduct object

An `S1MultiBurstProduct` contains a collection of Sentinel-1 burst products.

In [3]:
start_date = '2023-01-01'

reference_multiburst = asf.S1MultiBurstProduct(multiburst_group, start_date)

### Create an SBASNetwork from the S1MultiBurstProduct object

In [14]:
from datetime import datetime, date
import pandas as pd

def get_julian_season(season) -> tuple[int,int]:
    season_start_ts = pd.Timestamp(
        datetime.strptime(f"{season[0]}-0001", "%m-%d-%Y"), tz="UTC"
        )
    season_start_day = season_start_ts.timetuple().tm_yday
    season_end_ts = pd.Timestamp(
        datetime.strptime(f"{season[1]}-0001", "%m-%d-%Y"), tz="UTC"
    )
    season_end_day = season_end_ts.timetuple().tm_yday
    return (season_start_day, season_end_day)

season = ("1-1", "6-25")

multiburst_sbas = asf.SBASNetwork.from_geo_reference(
    geo_reference = reference_multiburst,
    start_date = '2023-01-01',
    end_date = '2025-10-02',
    season = get_julian_season(season),
    perpendicular_baseline=200, 
    inseason_temporal_baseline=36,
    bridge_target_date='3-1',
    bridge_year_threshold=1,
    allow_missing_state_vectors=True)

multiburst_sbas

# print(len(sbas.full_stack))

/Users/aflewandowski/Documents/asf_search/Discovery-asf_search/asf_search/ASFSearchOptions/ASFSearchOptions.py:126: UserWarning: While merging search options, existing option start:2023-01-01T00:00:00Z overwritten by kwarg with value 2023-01-01
  warnings.warn(msg)
/Users/aflewandowski/Documents/asf_search/Discovery-asf_search/asf_search/ASFSearchOptions/ASFSearchOptions.py:126: UserWarning: While merging search options, existing option end:2025-10-02T00:00:00Z overwritten by kwarg with value 2025-10-02
  warnings.warn(msg)
/Users/aflewandowski/Documents/asf_search/Discovery-asf_search/asf_search/ASFSearchOptions/ASFSearchOptions.py:126: UserWarning: While merging search options, existing option season:[1, 176] overwritten by kwarg with value (1, 176)
  warnings.warn(msg)


### Plot the `SBASNetwork`


In [16]:
multiburst_sbas.plot()

<hr>

## Add custom Pairs to the network.

### Create an S1MultiBurstSBASDelta object

`S1MultiBurstSBASDelta` objects map the `S1MultiBurstSBASNetwork.sbas_networks`' `geo_reference` products to a list of Pairs to add or remove to each `SBASNetwork.subset_stack` 

Below, we select Pairs from the remove lists by date using the `get_pair_from_dates` method. In this example, we create the S1MultiBurstSBASDelta object by finding pairs with the desired dates in each SBASNetwork's remove_list.

In [ ]:
try:
    from ciso8601 import parse_datetime
except ImportError:
    from dateutil.parser import parse as parse_datetime

burst_pair_collections = [asf.S1GeoReferenceBurstPairCollection(
    s.geo_reference, 
    [
        asf.get_pair_from_dates(s.remove_list, parse_datetime("20230304").date(), parse_datetime("20240614").date()),
        asf.get_pair_from_dates(s.remove_list, parse_datetime("20230127").date(), parse_datetime("20230304").date()),
    ])
    for s in multiburst_sbas.sbas_networks]

burst_pair_delta = asf.S1MultiBurstSBASDelta(burst_pair_collections, multiburst_sbas.sbas_networks)

### Add the identified pairs back to the SBASNetwork

In [ ]:
multiburst_sbas.add_pairs(burst_pair_delta)

### Plot the SBASNetwork

Note that the previously disconnected networks are now connected with a new pair (2023/03/04 - 2024/06/14). 

In [ ]:
multiburst_sbas.sbas_networks[0].plot()

### Add Pairs that are completly unknown to the SBASNetwork

Search for Pairs to add that were outside the bounds of the original SBASNetwork's time range and therefor are not on any remove_lists. 

In [ ]:
burst_pair_collections = []
for geo_ref in multiburst_sbas.geo_references:
    results = geo_ref.stack()
    burst_slcs = []
    for result in results:
        if parse_datetime(result.properties["startTime"]).date() == parse_datetime("2026-05-17").date() or \
            parse_datetime(result.properties["startTime"]).date() == parse_datetime("2026-05-23").date():
            burst_slcs.append(result)
    burst_pair_collections.append(asf.S1GeoReferenceBurstPairCollection(geo_ref, [asf.Pair(geo_ref, burst_slcs[0]), asf.Pair(geo_ref, burst_slcs[1]), asf.Pair(burst_slcs[0], burst_slcs[1])]))

burst_pair_delta = asf.S1MultiBurstSBASDelta(burst_pair_collections, multiburst_sbas.sbas_networks)

multiburst_sbas.add_pairs(burst_pair_delta)

### Plot the SBASNetwork again after adding more pairs

Notice that the SBASNetwork has expanded to include custom date pairs in 2026.

In [ ]:
multiburst_sbas.sbas_networks[0].plot()

<hr>

## Remove date pairs from the SBASNetwork

### Remove the first set of pairs we added

In [ ]:
multiburst_sbas.remove_pairs(burst_pair_delta)

### Plot the SBASNetwork, once again broken into two disconnected networks

In [ ]:
multiburst_sbas.sbas_networks[0].plot()

<hr>

## Use the `scene_ids` property to access multi-burst product IDs for easy InSAR product ordering from HyP3 

The `scene_ids` property provides scene IDs for the largest network in the `connected_substacks` list for each SBASNetwork in `sbas_networks`

An SBAS Network generated from a `S1MultiBurstProduct` geo-reference can be used to order Sentinel-1 multi-burst InSAR processing from ASF [HyP3](https://hyp3-docs.asf.alaska.edu/hyp3-docs/guides/burst_insar_product_guide/) and [HyP3+](https://hyp3-docs.asf.alaska.edu/hyp3-docs/about/hyp3_plus/)

In [ ]:
sbas_burst_ids = multiburst_sbas.scene_ids
sbas_burst_ids

<hr>

## Use the `get_scene_ids` method to access multi-burst product IDs from any of the SBASNetwork pair lists in `sbas_networks`

Define which pair lists to access using `asf_search.PairList`:
- `asf.PairList.SUBSET`: subset_stack
- `asf.PairList.REMOVE`: remove_list
- `asf.PairList.FULL`: full_stack
- `asf.Pairlist.CONNECTED`: connected_substacks



In [ ]:
# pass asf.PairList.SUBSET for IDs from the possible disconnected subset network
subset_burst_ids = multiburst_sbas.get_scene_ids(asf.PairList.SUBSET)
subset_burst_ids

In [ ]:
# pass asf.PairList.REMOVE for IDs from the remove_lists
remove_list_burst_ids = multiburst_sbas.get_scene_ids(asf.PairList.REMOVE)
remove_list_burst_ids

In [ ]:
# pass asf.PairList.CONNECTED and the connected_substack_index for IDs from a particular 
# network in connected_substacks
connected_substack_0_list_burst_ids = multiburst_sbas.get_scene_ids(asf.PairList.CONNECTED, connected_substack_index=0)
connected_substack_0_list_burst_ids